## Configuración Google Colab
> **Solo si estás en Colab:** ejecuta la siguiente celda para montar Google Drive.
> Si trabajas en local (VS Code / Jupyter), puedes saltártela.

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    import os
    # Replace 'archive' with the name of the folder you uploaded to Drive
    RUTA_DRIVE = '/content/drive/MyDrive/archive'
    os.chdir(RUTA_DRIVE)
    print(f"Directorio: {os.getcwd()}")
    # Install dependencies

    import subprocess
    subprocess.run(['pip', 'install', '-q', 'seaborn', 'scipy'])
else:
    import os
    print(f"Entorno local — directorio: {os.getcwd()}")

# EDA Proyecto I — Fase 3: Análisis Estadístico y Detección de Sesgos
**DataTalent Solutions S.L.** | Módulo II: Análisis y Visualización de Datos

En esta fase calculamos estadísticos descriptivos, exploramos correlaciones, analizamos grupos con `.groupby()` y — lo más crítico — **identificamos sesgos presentes en el dataset** con reflexión sobre su impacto en decisiones de negocio y modelos predictivos.

## 1. Carga de datos limpios

In [3]:
import pandas as pd
import numpy as np
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Load clean data
df     = pd.read_csv('data_roles_completo.csv', low_memory=False)
df_sal = pd.read_csv('data_roles_salario.csv',  low_memory=False)

print(f"Dataset completo (roles de datos): {df.shape}")
print(f"Dataset con salario limpio:        {df_sal.shape}")

Dataset completo (roles de datos): (19725, 50)
Dataset con salario limpio:        (6108, 50)


## 2. Estadística descriptiva — Variables clave

### 2.1 Salario anual (salary_annual)

In [4]:
s = df_sal['salary_annual']

print("=== Salario Anual (USD) ===")
print(f"  Recuento           : {s.count():>12,}")
print(f"  Media              : ${s.mean():>12,.0f}")
print(f"  Mediana (P50)      : ${s.median():>12,.0f}")
print(f"  Desviación estándar: ${s.std():>12,.0f}")
print(f"  Mínimo             : ${s.min():>12,.0f}")
print(f"  Percentil 25 (Q1)  : ${s.quantile(0.25):>12,.0f}")
print(f"  Percentil 75 (Q3)  : ${s.quantile(0.75):>12,.0f}")
print(f"  Máximo             : ${s.max():>12,.0f}")
print(f"  Asimetría (skew)   :  {s.skew():>12.3f}")
print(f"  Curtosis           :  {s.kurtosis():>12.3f}")

=== Salario Anual (USD) ===
  Recuento           :        6,108
  Media              : $     128,596
  Mediana (P50)      : $     124,800
  Desviación estándar: $      53,775
  Mínimo             : $      13,333
  Percentil 25 (Q1)  : $      90,000
  Percentil 75 (Q3)  : $     162,240
  Máximo             : $     281,000
  Asimetría (skew)   :         0.442
  Curtosis           :        -0.248


**Interpretación:** La diferencia positiva entre media y mediana confirma una **distribución con cola derecha**: existen algunos salarios muy altos que elevan la media por encima del valor típico. La asimetría positiva (skew > 0) lo cuantifica. Para orientar candidatos, la **mediana es el indicador más representativo** del salario que esperaría una persona con ese rol, ya que no está distorsionado por los valores extremos.

### 2.2 Vistas de oferta (views)

In [5]:
v = df['views']
print("=== Vistas por Oferta ===")
print(f"  Media              : {v.mean():>10,.1f}")
print(f"  Mediana (P50)      : {v.median():>10,.1f}")
print(f"  Desviación estándar: {v.std():>10,.1f}")
print(f"  P25                : {v.quantile(0.25):>10,.1f}")
print(f"  P75                : {v.quantile(0.75):>10,.1f}")
print(f"  Máximo             : {v.max():>10,.1f}")

=== Vistas por Oferta ===
  Media              :       22.7
  Mediana (P50)      :        5.0
  Desviación estándar:       74.0
  P25                :        4.0
  P75                :       15.0
  Máximo             :    5,132.0


**Interpretación:** La alta desviación estándar respecto a la media indica gran variabilidad: algunas ofertas reciben miles de vistas mientras la mayoría pasa desapercibida. Esto refleja el **efecto de reputación de marca**: las grandes empresas tech concentran la atención.

### 2.3 Solicitudes recibidas (applies)

In [6]:
a = df['applies']
print("=== Solicitudes por Oferta ===")
print(f"  Media              : {a.mean():>10,.1f}")
print(f"  Mediana (P50)      : {a.median():>10,.1f}")
print(f"  Desviación estándar: {a.std():>10,.1f}")
print(f"  P25                : {a.quantile(0.25):>10,.1f}")
print(f"  P75                : {a.quantile(0.75):>10,.1f}")
print(f"  Máximo             : {a.max():>10,.1f}")

=== Solicitudes por Oferta ===
  Media              :        7.5
  Mediana (P50)      :        4.0
  Desviación estándar:       20.7
  P25                :        4.0
  P75                :        4.0
  Máximo             :      566.0


**Interpretación:** La mediana de solicitudes es considerablemente más baja que la media, lo que confirma que pocas ofertas concentran la mayoría de las candidaturas (distribución de Pareto). Esto tiene implicaciones directas para DataTalent: los candidatos deben diferenciarse para destacar en las ofertas con mayor competencia.

## 3. Matriz de correlaciones (.corr())

In [7]:
num_cols = [c for c in ['salary_annual','views','applies','employee_count','follower_count']
            if c in df_sal.columns]

corr = df_sal[num_cols].corr()
print("Matriz de correlaciones (Pearson):")
print(corr.round(3).to_string())

Matriz de correlaciones (Pearson):
                salary_annual  views  applies  employee_count  follower_count
salary_annual           1.000  0.014    0.003           0.192           0.211
views                   0.014  1.000    0.857          -0.020          -0.005
applies                 0.003  0.857    1.000          -0.041          -0.024
employee_count          0.192 -0.020   -0.041           1.000           0.877
follower_count          0.211 -0.005   -0.024           0.877           1.000


**Interpretación de la matriz de correlaciones:**
- **`views` ↔ `applies`:** correlación positiva esperada — más visibilidad genera más candidaturas.
- **`salary_annual` ↔ `employee_count`:** si positiva, las grandes empresas pagan más (economías de escala y presupuestos mayores).
- **`follower_count` ↔ `views`:** empresas con más seguidores en LinkedIn atraen más tráfico a sus ofertas.
- **`salary_annual` ↔ `views`:** correlación cercana a 0 indica que publicar el salario no determina significativamente la visibilidad de la oferta.

Nota: correlación no implica causalidad. Estos valores cuantifican relaciones lineales; relaciones no lineales pueden existir y no quedar capturadas aquí.

## 4. Análisis por grupos (.groupby() y pivot_table())

### 4.0 Índice de Gini

El Índice de Gini es una métrica que se utiliza para medir el nivel de desigualdad o de concentración de un recurso (como los salarios) dentro de una población determinada.

A diferencia de la desviación típica, que mide distancias absolutas, el Índice de Gini mide proporciones relativas.

El índice está acotado entre 0 y 1:

* Gini = 0 (Igualdad perfecta): Significa que el recurso está repartido de forma idéntica entre todos.

* Gini = 1 (Desigualdad perfecta): Significa que una sola persona concentra la totalidad del recurso, mientras que el resto no tiene nada.

Si se oredena los datos de manera ascendente, de modo que $x_1 \le x_2 \le \dots \le x_n$, la fórmula del Índice de Gini para una muestra es:

$$
G = \frac{\sum_{i=1}^{n} (2i - n - 1) \cdot x_i}{n \cdot \sum_{i=1}^{n} x_i}
$$

Donde:
- $n$: El tamaño de la muestra.
- $x_i$: El elemento en la posición $i$ de la muestra (una vez ordenados de menor a mayor).
- $\sum_{i=1}^{n} x_i$: La suma total de todos los elementos de la muestra.
- $(2i - n - 1)$: Es el factor de ponderación que da más peso a los elementos que se encuentran en las posiciones más altas de la muestra ordenada.

In [25]:
def gini_coef(series):
    """
    Calculates the Gini Coefficient for a single Pandas Series.
    Automatically drops missing values (NaN) and values <= 0.

    Parameters:
    series (pd.Series): The Pandas Series containing the numerical data.

    Returns:
    float: The Gini coefficient (between 0 and 1).
    """
    # Extract values, drop NaNs, and filter out non-positive values
    values = series.dropna().values
    values = values[values > 0]

    if len(values) == 0:
        raise ValueError("The series does not contain enough valid values greater than 0.")

    # Sort values in ascending order
    sorted_values = np.sort(values)
    n = len(sorted_values)

    # Calculate Gini coefficient
    index = np.arange(1, n + 1)
    return (2 * np.sum(index * sorted_values) / (n * np.sum(sorted_values))) - ((n + 1) / n)

### 4.1 Salario por nivel de experiencia

In [38]:
if 'formatted_experience_level' in df_sal.columns:
    exp_sal = (df_sal.groupby('formatted_experience_level')['salary_annual']
               .agg(Media='mean',
                    Mediana='median',
                    N_ofertas='count',
                    Std='std',
                    Gini_Coef=gini_coef)
               .round({'Media': 0, 'Mediana': 0, 'Std': 0, 'Gini_Coef': 3})
               .sort_values('Mediana', ascending=False))
    print("Salario anual por nivel de experiencia:")
    print(exp_sal.to_string())

Salario anual por nivel de experiencia:
                               Media   Mediana  N_ofertas      Std  Gini_Coef
formatted_experience_level                                                   
director                    209560.0  210000.0         86  47217.0      0.124
executive                   176583.0  181100.0         14  80565.0      0.250
mid-senior level            138529.0  135000.0       4051  51967.0      0.212
entry level                 104767.0   99840.0       1222  50474.0      0.270
associate                   105743.0   99070.0        687  39574.0      0.205
internship                   64952.0   62400.0         48  21306.0      0.161


**Interpretación:** La escalera salarial (Entry < Mid-Senior < Director < Executive) es una de las variables más importantes para el diseño del programa de reskilling: cuantifica exactamente cuánto valor económico aporta cada nivel de cualificación. Si la diferencia entre Entry y Mid-Senior es grande, el ROI del reskilling es alto y justifica la inversión.

**Observaciones**:

1. *El fenómeno de los Juniors* (`entry level` vs `associate`): Si solo se tiene en cuenta la mediana, un perfil entry level ($99.840\$$) gana prácticamente lo mismo que un associate ($99.070\$$). Sin embargo, sus estructuras de mercado son radicalmente opuestas: el grupo entry level tiene el Gini más alto de la tabla (0.270) y una desviación estándar muy alta ($50.474\$$). Esto significa que el mercado de primer empleo está muy polarizado. Hay ofertas junior en empresas locales o pequeñas con salarios muy bajos, pero también hay empresas (como las Big Tech o FAANG) que pagan salarios muy competitivos a los recién graduados con talento. Hay mucha desigualdad en este nivel. El grupo associate tiene un Gini mucho más bajo (0.205) y menor dispersión ($39.574\$$). Esto indica que una vez el profesional pasa los primeros años de experiencia, el mercado se estandariza. Las bandas salariales corporativas para puestos intermedios están mucho más reguladas y acotadas.

2. *La paradoja de los directores* (`director`): Los directores tienen la desviación típica más alta en dólares ($47.217\$$), lo cual es lógico porque sus sueldos son muy altos. Pero tienen el coeficiente de Gini más bajo de todo el dataset (0.124). Lo que significa que, aunque los salarios varían en miles de dólares en términos absolutos, relativamente el mercado directivo es el más equitativo de todos. Casi ningún director se despega de la masa; todos ganan cifras muy similares dentro de una global salarial alto. Hay una banda estándar de dirección muy consolidada en LinkedIn.

3. *La volatilidad de los ejecutivos* (`executive`): Aunque la muestra es muy pequeña ($n=14$), de los datos se oberva que con un Gini de 0.250 y una Std de $80.565\$$, en el nivel ejecutivo hay una dispersión muy grande, los salarios de CEOs, CTOs, CDOs, etc., no siguen ninguna regla de mercado: dependen del tamaño de la empresa que contrata y de paquetes de compensación variable.  

### 4.2 Número de ofertas y salario por tipo de contrato

In [39]:
wt_sal = (df_sal.groupby('formatted_work_type')['salary_annual']
           .agg(Media='mean',
                Mediana='median',
                N_ofertas='count',
                Std='std',
                Gini_Coef=gini_coef)
           .round({'Media': 0, 'Mediana': 0, 'Std': 0, 'Gini_Coef': 3})
           .sort_values('N_ofertas', ascending=False))
print("Por tipo de contrato:")
print(wt_sal.to_string())

Por tipo de contrato:
                        Media   Mediana  N_ofertas      Std  Gini_Coef
formatted_work_type                                                   
full-time            132681.0  125000.0       4823  54683.0      0.233
contract             120858.0  120640.0       1043  42169.0      0.198
part-time             64809.0   40768.0        138  47665.0      0.337
internship            73198.0   57720.0         45  43641.0      0.263
temporary            113988.0  107744.0         37  54100.0      0.266
other                137970.0  135250.0         22  41278.0      0.165


**Observaciones**:
1. *La anomalía del empleo a tiempo parcial* (`part-time`): Tiene la mediana más baja de todo el dataset ($40.768\$$), pero una desviación estándar proporcionalmente muy alta ($47.665\$$). Además, el coeficiente de Gini es el más alto de la tabla (0.337). Esto lleva a pensar que que el empleo part-time en el mundo de los datos está muy polarizado. Por un lado, hay ofertas muy precarias o de soporte con sueldos extremadamente bajos (que llevan la mediana hacia abajo). Por otro lado, hay ofertas específicas de consultoría externa por horas que están muy bien remuneradas, elevando la desigualdad interna de este grupo.

2. *La estabilidad estructural de los autónomos* (`contract`): Tradicionalmente se piensa que los autónomos se mueven en un mercado más variable. En los datos de LinkedIn se ve lo contrario:
- Tienen un Gini bastante bajo (0.198) y una Std muy controlada ($42.169\$$).
- Además, su media y mediana son casi iguales($120.858\$$ vs $120.640\$$), lo que indica una distribución casi simétrica. El mercado de autónomos de datos está bastante estandarizado. Las empresas saben cuánto se paga por hora/proyecto, configurando un entorno predecible y muy homogéneo.
3. *El peso de la masa salarial*: El segmento `full-time` concentra la mayoría del dataset con 4,823 ofertas. Su Gini es de 0.233, lo que indica una distribución salarial bastante equitativa. Existe una dispersión lógica provocada por los saltos de experiencia (de Junior a Senior), pero el mercado de contratación indefinida mantiene una estructura de bandas corporativas estables y consistentes.

### 4.3 Top industrias por ofertas y salario

In [40]:
if 'job_industries_list' in df_sal.columns:
    df_ind = df_sal[['job_id','salary_annual','job_industries_list']].dropna(subset=['job_industries_list']).copy()
    df_ind['industry'] = df_ind['job_industries_list'].str.split(', ')
    df_ind_exp = df_ind.explode('industry')

    ind_stats = (df_ind_exp.groupby('industry')['salary_annual']
                 .agg(Media='mean',
                      N_ofertas='count',
                      Std='std',
                      Gini_Coef=gini_coef)
                 .round({'Media': 0, 'Std': 0, 'Gini_Coef': 3})
                 .sort_values('N_ofertas', ascending=False)
                 .head(15))
    print("Top 15 industrias — roles de datos con salario:")
    print(ind_stats.to_string())

Top 15 industrias — roles de datos con salario:
                                    Media  N_ofertas      Std  Gini_Coef
industry                                                                
IT Services and IT Consulting    140378.0        780  53652.0      0.214
Software Development             166474.0        536  58125.0      0.200
Financial Services               140980.0        536  49492.0      0.199
Staffing and Recruiting          118125.0        467  43842.0      0.210
Hospitals and Health Care        122097.0        319  51258.0      0.233
Technology                       163887.0        281  55478.0      0.193
Banking                          131346.0        254  46982.0      0.201
Manufacturing                    104873.0        244  38823.0      0.203
Defense and Space Manufacturing  161143.0        214  50274.0      0.178
Medical Equipment Manufacturing  115802.0        195  49240.0      0.239
Pharmaceutical Manufacturing     126825.0        153  48385.0      0.212
Inf

**Observaciones**:

- Aunque los salarios medios presentan diferencias importantes la mayoría de sectores tienen un Gini parecido y relativamente bajo ($\approx 0.200$). Esto indica mercados muy estandarizados. Las bandas salariales para roles de datos están muy consolidadas; la progresión es predecible y homogénea.
- Los sectores de *IT Services*, *Software* y *Finance* dominan la demanda de empleo, acumulando juntas casi 1,850 ofertas con salario publicado.
- *Information & Internet*: Registra la media salarial más alta de todo el ranking ($178,738\$$) combinada con uno de los coeficientes de Gini más bajos (0.178).
- *Defense and Space Manufacturing*: Remunera muy bien ($161,143) los puestos de datos y tiene un Gini muy bajo (0.178) con una desviación estándar contenida. Al ser una industria fuertemente regulada (contratos gubernamentales, requerimientos estrictos de seguridad y certificaciones del personal), las bandas salariales son rígidas y muy altas por decreto, dejando poco margen a la desigualdad interna.
- El contrapunto: Las industrias médica y de salud. *Medical Equipment Manufacturing* (Gini: 0.239) y *Hospitals and Health Care* (Gini: 0.233): Son los entornos con mayor desigualdad relativa de toda la muestra. En salud y equipamiento médico, los datos están polarizados, coexisten puestos de analistas de datos clínicos o gestores de registros básicos con sueldos modestos, junto a científicos de datos especializados en bioinformática, visión por computadora para diagnóstico o IA médica que perciben ingresos mucho más alto, ensanchando la brecha del sector.

### 4.4 Top skills más demandadas

In [11]:
if 'job_skills_list' in df.columns:
    df_sk = df[['job_id','job_skills_list']].dropna(subset=['job_skills_list']).copy()
    df_sk['skill'] = df_sk['job_skills_list'].str.split(', ')
    df_sk_exp = df_sk.explode('skill')
    skill_counts = df_sk_exp['skill'].value_counts().head(15)
    print("Top 15 skills más demandadas en roles de datos:")
    for sk, cnt in skill_counts.items():
        pct = cnt / len(df) * 100
        print(f"  {str(sk):<35} {cnt:>6,}  ({pct:.1f}%)")

Top 15 skills más demandadas en roles de datos:
  Information Technology              10,471  (53.1%)
  Engineering                          7,749  (39.3%)
  Manufacturing                        2,179  (11.0%)
  Management                           1,962  (9.9%)
  Analyst                              1,718  (8.7%)
  Sales                                1,405  (7.1%)
  Finance                              1,159  (5.9%)
  Research                             1,128  (5.7%)
  Business Development                   798  (4.0%)
  Other                                  794  (4.0%)
  Quality Assurance                      535  (2.7%)
  Project Management                     485  (2.5%)
  Consulting                             466  (2.4%)
  Accounting/Auditing                    449  (2.3%)
  Design                                 350  (1.8%)


**Interpretación:** Las skills más demandadas definen directamente el currículo del programa de reskilling. Una persona que domine el top 5 ya cubre la mayoría de las ofertas del mercado. Nótese que las 35 categorías del dataset son amplias (no específicas como "Python" o "SQL"), lo que significa que cada categoría engloba múltiples herramientas concretas.

### 4.5 Tabla pivot: mediana salarial por experiencia × tipo de contrato

In [44]:
# Define seniority and contract-types order
seniority_order = ['executive', 'director', 'mid-senior level', 'associate', 'entry level', 'internship']
contract_types_order = ['full-time', 'contract', 'part-time', 'temporary', 'internship', 'other']

try:
    pivot = pd.pivot_table(
        df_sal,
        values='salary_annual',
        index='formatted_experience_level',
        columns='formatted_work_type',
        aggfunc='median',
        fill_value=0
    ).round(0)
    # Reindex both axes
    ordered_pivot = pivot.reindex(index=seniority_order, columns=contract_types_order, fill_value=0)
    print("Tabla pivot — Mediana salarial (USD) por experiencia y tipo de contrato:")
    print(ordered_pivot.to_string())
except Exception as e:
    print(f"Nota: {e}")

Tabla pivot — Mediana salarial (USD) por experiencia y tipo de contrato:
formatted_work_type         full-time  contract  part-time  temporary  internship     other
formatted_experience_level                                                                 
executive                    175000.0       0.0        0.0   187200.0         0.0       0.0
director                     213100.0  178800.0        0.0        0.0     49920.0       0.0
mid-senior level             135200.0  133120.0    39520.0   105872.0     52000.0  104000.0
associate                    105000.0   84240.0   103700.0   124800.0         0.0       0.0
entry level                  100000.0   89440.0    43576.0    98800.0     76586.0  138900.0
internship                    64608.0   60320.0    62400.0    47840.0     56940.0       0.0


## 5. Probabilidad condicional P(A|B)

In [13]:
# P(nivel senior | rol Data Scientist) vs P(senior | Data Engineer) vs P(senior | global)
df_p = df.copy()
df_p['is_senior'] = df_p['formatted_experience_level'].str.contains(
    'senior|director|executive', case=False, na=False
)
df_p['is_ds'] = df_p['title'].str.contains('scientist|science', case=False, na=False)
df_p['is_de'] = df_p['title'].str.contains('data engineer', case=False, na=False)
df_p['is_da'] = df_p['title'].str.contains('data analyst', case=False, na=False)

p_s_ds  = df_p.loc[df_p['is_ds'], 'is_senior'].mean()
p_s_de  = df_p.loc[df_p['is_de'], 'is_senior'].mean()
p_s_da  = df_p.loc[df_p['is_da'], 'is_senior'].mean()
p_s_all = df_p['is_senior'].mean()

print("=== Probabilidad Condicional — P(Senior | Rol) ===")
print(f"  P(Senior | Data Scientist) = {p_s_ds:.3f}  ({p_s_ds*100:.1f}%)")
print(f"  P(Senior | Data Engineer)  = {p_s_de:.3f}  ({p_s_de*100:.1f}%)")
print(f"  P(Senior | Data Analyst)   = {p_s_da:.3f}  ({p_s_da*100:.1f}%)")
print(f"  P(Senior | cualquier rol)  = {p_s_all:.3f}  ({p_s_all*100:.1f}%)")

=== Probabilidad Condicional — P(Senior | Rol) ===
  P(Senior | Data Scientist) = 0.780  (78.0%)
  P(Senior | Data Engineer)  = 0.820  (82.0%)
  P(Senior | Data Analyst)   = 0.615  (61.5%)
  P(Senior | cualquier rol)  = 0.677  (67.7%)


**Interpretación:** La probabilidad condicional P(Senior | Data Scientist) nos dice qué porcentaje de las ofertas de Data Scientist requieren nivel senior. Comparada con la probabilidad marginal P(Senior), revela si el rol demanda más experiencia que la media. Esto es información concreta para que DataTalent diseñe rutas de reskilling realistas: no tiene sentido orientar candidatos junior a posiciones que estadísticamente exigen senior.

## 6. Detección de Sesgos en el Dataset

Los sesgos en los datos pueden llevar a decisiones empresariales erróneas y a modelos de IA discriminatorios. Identificamos **4 sesgos** con su origen y su impacto potencial.

### Sesgo 1: MNAR en datos salariales (*Missing Not At Random*)
**Descripción:** Las columnas salariales tienen entre el 71% y el 95% de valores nulos. Los datos no faltan aleatoriamente: existe una razón sistemática para que falten.

**Origen:** Las empresas eligen estratégicamente si publicar el salario. Las startups y PYMEs con salarios menos competitivos suelen ocultarlos para no desalentar candidatos.


In [14]:
total = len(df)
con_salario = df['salary_annual'].notna().sum()
sin_salario = total - con_salario

print("=== Análisis MNAR — Datos Salariales ===")
print(f"Con salario publicado:    {con_salario:>7,}  ({con_salario/total*100:.1f}%)")
print(f"Sin salario publicado:    {sin_salario:>7,}  ({sin_salario/total*100:.1f}%)")
print()
# Do larger companies publish salaries more often?
if 'comp_size' in df.columns:
    print("Tasa de publicación de salario por tamaño de empresa:")
    res = df.groupby('comp_size')['salary_annual'].apply(
        lambda x: f"{x.notna().mean()*100:.0f}%  ({x.notna().sum():,} de {len(x):,})"
    )
    print(res.to_string())

=== Análisis MNAR — Datos Salariales ===
Con salario publicado:      6,355  (32.2%)
Sin salario publicado:     13,370  (67.8%)

Tasa de publicación de salario por tamaño de empresa:
comp_size
1.0      32%  (511 de 1,596)
2.0      30%  (713 de 2,349)
3.0      34%  (582 de 1,725)
4.0      34%  (504 de 1,503)
5.0    33%  (1,290 de 3,940)
6.0      28%  (456 de 1,614)
7.0    32%  (1,987 de 6,182)


**Impacto en un modelo predictivo:** Un modelo entrenado solo con los datos con salario aprendería los salarios de empresas que voluntariamente los publican (generalmente las más grandes y con mejor remuneración). Consecuencia: **sobreestimaría sistemáticamente** los salarios del mercado real. DataTalent podría orientar a candidatos con expectativas infladas, generando frustración en el proceso de selección.

### Sesgo 2: Sesgo geográfico — El dataset no representa el mercado español
**Descripción:** El dataset proviene de LinkedIn global, con predominancia de ofertas en EEUU. Si se usa para orientar candidatos en España, los hallazgos salariales y de skills no son directamente aplicables.

**Origen:** El scraping de LinkedIn se realizó principalmente en mercados anglosajones (mayor penetración de LinkedIn). Las ofertas españolas están subrepresentadas.

In [15]:
if 'comp_country' in df.columns:
    country_counts = df['comp_country'].value_counts().head(12)
    total_con_pais = df['comp_country'].notna().sum()
    print(f"Distribución geográfica (de {total_con_pais:,} registros con país):\n")
    for country, cnt in country_counts.items():
        pct = cnt / total_con_pais * 100
        bar = '█' * int(pct / 2)
        print(f"  {str(country):<30} {cnt:>6,}  ({pct:5.1f}%)  {bar}")
    print(f"\nRegistros sin país: {df['comp_country'].isna().sum():,}")

Distribución geográfica (de 19,556 registros con país):

  us                             17,090  ( 87.4%)  ███████████████████████████████████████████
  gb                                744  (  3.8%)  █
  0                                 407  (  2.1%)  █
  in                                263  (  1.3%)  
  ca                                181  (  0.9%)  
  se                                133  (  0.7%)  
  de                                116  (  0.6%)  
  ch                                100  (  0.5%)  
  no                                 78  (  0.4%)  
  fr                                 76  (  0.4%)  
  nl                                 56  (  0.3%)  
  oo                                 51  (  0.3%)  

Registros sin país: 169


**Impacto en un modelo predictivo:** Un modelo entrenado con este dataset y aplicado al mercado español **transladaría estándares salariales estadounidenses** (generalmente mucho más altos) a un contexto europeo, generando expectativas irreales. Las skills demandadas también pueden diferir: el mercado español tiene mayor presencia de banca, turismo y sector público.

**Medida de mitigación:** Complementar con datos de Infojobs, Tecnoempleo o SEPE para el contexto español.

### Sesgo 3: Sesgo de selección — Solo ofertas publicadas en LinkedIn
**Descripción:** LinkedIn es dominante para perfiles tech, pero no representa el mercado completo. Infojobs, Indeed, portales gubernamentales y ofertas directas (sin plataforma) quedan excluidos.

**Origen:** El dataset se extrajo de una única fuente, creando un sesgo estructural hacia empresas con marca empleadora activa en LinkedIn.

In [16]:
print("=== Sesgo de Selección — Fuente única ===")
print("Plataforma: LinkedIn Job Postings (Kaggle)")
print()
print("Empresas sobrerepresentadas: grandes tecnológicas con marca empleadora fuerte")
print("Empresas subrepresentadas:  PYMEs, sector público, consultoras pequeñas")
print()
if 'comp_size' in df.columns:
    print("Distribución de ofertas por tamaño de empresa:")
    cs = df['comp_size'].value_counts(dropna=False)
    for size, cnt in cs.items():
        pct = cnt / len(df) * 100
        print(f"  {str(size):<15} {cnt:>7,}  ({pct:.1f}%)")

=== Sesgo de Selección — Fuente única ===
Plataforma: LinkedIn Job Postings (Kaggle)

Empresas sobrerepresentadas: grandes tecnológicas con marca empleadora fuerte
Empresas subrepresentadas:  PYMEs, sector público, consultoras pequeñas

Distribución de ofertas por tamaño de empresa:
  7.0               6,182  (31.3%)
  5.0               3,940  (20.0%)
  2.0               2,349  (11.9%)
  3.0               1,725  (8.7%)
  6.0               1,614  (8.2%)
  1.0               1,596  (8.1%)
  4.0               1,503  (7.6%)
  nan                 816  (4.1%)


**Impacto en un modelo predictivo:** El modelo aprendería que "una oferta de datos típica proviene de una gran empresa tech con perfil LinkedIn activo". Fallaría en predecir salarios y skills para roles en PYMEs o sector público, que representan una parte significativa del tejido empresarial español (especialmente relevante para candidatos de ciudades no metropolitanas).

### Sesgo 4: Ausencia de datos de género (*Undisclosed protected attribute*)
**Descripción:** El dataset no contiene información de género. Sin este dato, es imposible detectar directamente brechas salariales de género, uno de los análisis más críticos para garantizar equidad en el reskilling.

**Origen:** LinkedIn no incluye género por defecto en sus datos de ofertas de empleo. Incorporarlo requeriría autodeclaración o inferencia algorítmica, ambas con problemas legales y éticos bajo el RGPD europeo.

In [17]:
print("=== Sesgo por Atributo Protegido Ausente (Género) ===")
print()
print("Variables de género: NO disponibles en el dataset")
print()
print("Consecuencias directas:")
print("  1. No podemos cuantificar la brecha salarial de género")
print("  2. Un modelo entrenado sin control de género puede perpetuar sesgos históricos")
print("  3. Las decisiones de reskilling sin perspectiva de género pueden ser excluyentes")
print()
# Proxy: distribution of roles that have historically shown gender bias
if 'title' in df.columns:
    tech_roles = df['title'].str.contains('engineer|developer|architect', case=False, na=False).sum()
    soft_roles = df['title'].str.contains('analyst|coordinator|manager', case=False, na=False).sum()
    print(f"Roles de ingeniería/desarrollo (históricamente masculinizados): {tech_roles:,} ({tech_roles/len(df)*100:.1f}%)")
    print(f"Roles analíticos/gestión (mayor diversidad de género): {soft_roles:,}  ({soft_roles/len(df)*100:.1f}%)")
    print()
    print("Nota: esta es una aproximación indirecta, no un análisis de género real.")

=== Sesgo por Atributo Protegido Ausente (Género) ===

Variables de género: NO disponibles en el dataset

Consecuencias directas:
  1. No podemos cuantificar la brecha salarial de género
  2. Un modelo entrenado sin control de género puede perpetuar sesgos históricos
  3. Las decisiones de reskilling sin perspectiva de género pueden ser excluyentes

Roles de ingeniería/desarrollo (históricamente masculinizados): 11,490 (58.3%)
Roles analíticos/gestión (mayor diversidad de género): 5,905  (29.9%)

Nota: esta es una aproximación indirecta, no un análisis de género real.


**Impacto en un modelo predictivo:** Un modelo de predicción salarial sin control de género puede aprender patrones discriminatorios implícitos: si históricamente las mujeres están subrepresentadas en roles senior de ingeniería de datos, el modelo aprenderá a predecir salarios más bajos para ciertos patrones de carrera. Esto infringiría el principio de equidad algorítmica (Fairness AI) y podría vulnerar legislación anti-discriminación europea.

**Medida de mitigación:** Cruzar con encuestas que incluyan autodeclaración de género (Stack Overflow Developer Survey, Kaggle ML Survey) para análisis de equidad de género.

## 7. Resumen del análisis estadístico

| Hallazgo | Métrica clave | Implicación para DataTalent |
|----------|-------------|----------------------------|
| Salario mediano en roles de datos | Ver análisis § 2.1 | Referencia para orientación salarial realista |
| Skills más demandadas | Top 5 del ranking | Define el currículo mínimo del programa de reskilling |
| ROI del nivel de experiencia | Diferencia median Entry vs Senior | Argumento económico para justificar la inversión en formación |
| Industrias con más demanda | Top 3 de § 4.3 | Sectores objetivo para colocación de candidatos |
| **MNAR salarial** | 71–95% de nulos | Las cifras salariales son una estimación parcial sesgada hacia arriba |
| **Sesgo geográfico** | Mayoría EEUU | Ajustar o complementar con datos españoles antes de usar |
| **Sin datos de género** | 0% cobertura | Requiere fuente externa para garantizar equidad en el programa |

**Próximo paso (Fase 4):** Visualización de todos estos hallazgos.